In [1]:
import os
import numpy as np
import open3d as o3d

data_dir = "pcd_data/"

frames = []
pcds = []

##### manual

# tf_path = os.path.join(data_dir, "view01_tf.npy")
# tf = np.load(tf_path)
# tf_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0)
# tf_frame.transform(tf)
# frames.append(tf_frame)

##### end of manual


# ---------------------------------
# Load all tf files automatically
# ---------------------------------

tf_files = [f for f in os.listdir(data_dir) if f.endswith("_tf.npy")]
tf_files.sort()

print("Found TF files:", tf_files)

for tf_file in tf_files:

    tf_path = os.path.join(data_dir, tf_file)

    # load transformation
    tf = np.load(tf_path)

    # create coordinate frame
    tf_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0)
    tf_frame.transform(tf)
    frames.append(tf_frame)

    # create PCD
    pcd_name = tf_file.replace("_tf.npy", ".pcd")
    pcd_path = os.path.join(data_dir, pcd_name)
    pcd = o3d.io.read_point_cloud(pcd_path)
    pcds.append(pcd)
    



obj_pose_path = os.path.join(data_dir, "initial_obj_pose.npy")
tf_obj = np.load(obj_pose_path)
object_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0, origin=tf_obj[:3])
frames.append(object_frame)


world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0, origin=[0, 0, 0])
frames.append(world_frame)



# Source Mesh
translation = tf_obj[:3]
rotation_deg = tf_obj[3:]
rotation_rad = np.radians(rotation_deg)
T = np.eye(4)
R = o3d.geometry.get_rotation_matrix_from_xyz(rotation_rad)
T[:3, :3] = R
T[:3, 3] = translation

SOURCE_PATH = "workpiece/first_object.stl" #  CAD model
mesh = o3d.io.read_triangle_mesh(SOURCE_PATH)
mesh.compute_vertex_normals() # Makes the shading look better
R_mat = mesh.get_rotation_matrix_from_axis_angle([0, 0, np.radians(90)])
mesh.rotate(R_mat, center=[0, 0, 0])
mesh.transform(T)


# pcd_path = os.path.join(data_dir, "view01.pcd")
# pcd = o3d.io.read_point_cloud(pcd_path)


o3d.visualization.draw_geometries(frames + pcds)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Found TF files: ['view01_tf.npy', 'view02_tf.npy', 'view03_tf.npy', 'view04_tf.npy', 'view05_tf.npy', 'view06_tf.npy', 'view07_tf.npy', 'view08_tf.npy', 'view09_tf.npy', 'view10_tf.npy', 'view11_tf.npy', 'view12_tf.npy', 'view13_tf.npy', 'view14_tf.npy', 'view15_tf.npy', 'view16_tf.npy', 'view17_tf.npy', 'view18_tf.npy', 'view19_tf.npy']
[Open3D WARNING] Read PCD failed: unable to open file: pcd_data\view02.pcd
[Open3D WARNING] Read PCD failed: unable to open file: pcd_data\view03.pcd
[Open3D WARNING] Read PCD failed: unable to open file: pcd_data\view04.pcd
[Open3D WARNING] Read PCD failed: unable to open file: pcd_data\view05.pcd
[Open3D WARNING] Read PCD failed: unable to open file: pcd_data\view06.pcd
[Open3D WARNING] Read PCD failed: unable to open file: pcd_data\view07.pcd
[Open3D WARNING] Read PCD failed: unable 

In [54]:
# Helper Functions

import os
import math
import copy
import open3d as o3d
import numpy as np
from scipy.spatial.transform import Rotation as R



def get_rotation_matrix_z(deg):
    """Creates a 3x3 rotation matrix for the Z-axis."""
    rad = math.radians(deg)
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[c, -s, 0], 
                     [s,  c, 0], 
                     [0,  0, 1]])


def process_point_cloud(pcd, initial_pos, initial_angle, box_size):
    """Processes a single PCD file: removes ground plane and crops to OBB."""
    distance_threshold = 3.0 # mm, for RANSAC plane segmentation
    ransac_n = 3
    num_iterations = 2000
    plane_offset = 0.5 # mm, to ensure we keep points just above the plane

    # 1. Load the specific file    
    if pcd.is_empty():
        print(f"Warning: Point cloud is empty or not found.")
        return pcd

    # 2. Define the Oriented Bounding Box (OBB)
    # Using your rotation logic to align with the YOLO/Initial guess
    rot_matrix = get_rotation_matrix_z(-initial_angle) 
    obb = o3d.geometry.OrientedBoundingBox(
        center=np.array(initial_pos), 
        R=rot_matrix, 
        extent=np.array(box_size)
    )

    # 3. Detect and remove the ground plane (RANSAC)
    # distance_threshold=3.0 is quite liberal; adjust if it cuts your object
    plane_model, inliers = pcd.segment_plane(distance_threshold=distance_threshold, 
                                             ransac_n=ransac_n, 
                                             num_iterations=num_iterations)
    [a, b, c, d] = plane_model

    # 4. Filter points above the plane
    pts = np.asarray(pcd.points)
    # ax + by + cz + d > offset
    distances = a * pts[:, 0] + b * pts[:, 1] + c * pts[:, 2] + d
    above_plane_indices = np.where(distances > plane_offset)[0] 
    pcd_filtered = pcd.select_by_index(above_plane_indices)

    # 5. Crop to the region of interest
    final_pcd = pcd_filtered.crop(obb)

    return final_pcd


def load_viewpoint_data(index, actual_dir, sim_dir):
    """
    Loads actual pcd, simulated pcd, and the pose matrix for a specific index.
    
    Args:
        index (int): The viewpoint index (e.g., 0, 1, 2)
        actual_dir (str): Path to 'pcd_data' folder
        sim_dir (str): Path to 'viewpoints_candidate' folder
        
    Returns:
        actual_pcd, sim_pcd, pose_matrix
    """
    # 1. Format the filenames
    # Actual uses 2-digit padding: view00.pcd, view01.pcd
    actual_name = f"view{index:02d}.pcd"
    # Simulated uses raw index: viewpoint_simulated_0.pcd
    sim_name = f"viewpoint_simulated_{index}.pcd"
    # Pose uses raw index: viewpoint_pose_0.npy
    pose_name = f"viewpoint_pose_{index}.npy"

    # 2. Build Full Paths
    actual_path = os.path.join(actual_dir, actual_name)
    sim_path = os.path.join(sim_dir, sim_name)

    # 3. Load Data
    try:
        actual_pcd = o3d.io.read_point_cloud(actual_path)
        sim_pcd = o3d.io.read_point_cloud(sim_path)
        
        return actual_pcd, sim_pcd 
        
    except Exception as e:
        print(f"Error loading index {index}: {e}")
        return None, None


def pose_to_matrix(pose):
    """
    Converts [x, y, z, roll, pitch, yaw] in degrees to a 4x4 transformation matrix.
    If zyz=True, assumes the input is in ZYZ Euler Angles, otherwise XYZ Euler Angles.
    
    Args:
    - pose (list or array): [x, y, z, roll, pitch, yaw] in degrees.
    - zyz (bool): If True, interprets the last three values as ZYZ Euler angles. 
                  If False, uses XYZ Euler angles (default).
    
    Returns:
    - T (ndarray): The 4x4 homogeneous transformation matrix.
    """
    x, y, z = pose[:3]   # Translation vector
    roll, pitch, yaw = pose[3:]  # Rotation angles in degrees
    
    # Convert XYZ Euler Angles to a 3x3 rotation matrix
    rot_matrix = R.from_euler('xyz', [roll, pitch, yaw], degrees=True).as_matrix()

    # Build the 4x4 Homogeneous Transformation Matrix
    T = np.eye(4)  # Start with the identity matrix (4x4)
    T[:3, :3] = rot_matrix  # Set the upper-left 3x3 part to the rotation matrix
    T[:3, 3] = [x, y, z]    # Set the upper-right 3x1 part to the translation vector
    return T


def preprocess_normal(pcd, num_points=False, invert_normals=False):
    """Downsamples, estimates normals, and computes FPFH features."""
    if num_points:
        pcd_down = pcd.farthest_point_down_sample(num_points)
    else:
        pcd_down = pcd
    avg_dist = np.mean(pcd_down.compute_nearest_neighbor_distance())
    
    # Estimate Normals
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * 2, max_nn=30))
    
    # Orientation fix: Ensure normals point 'up' (positive Z)
    normals = np.asarray(pcd_down.normals)
    
    if invert_normals:
        for i in range(len(normals)):
            if normals[i][2] < 0:
                normals[i] *= -1

    # Compute FPFH (Feature descriptors for global matching)
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down, o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * 5, max_nn=100))
    
    return pcd_down, fpfh


def run_global_registration(source, target, source_fpfh, target_fpfh, voxel_size):
    """
    Performs RANSAC-based global registration to find a rough alignment.
    """
    distance_threshold = voxel_size * 1.5
    
    # RANSAC based on feature matching
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source, target, source_fpfh, target_fpfh, 
        mutual_filter=True,
        max_correspondence_distance=distance_threshold,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=3, 
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
        ], 
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500)
    )
    return result


def run_global_registration_adaptive(source_down, target_down, source_fpfh, target_fpfh):
    """
    Performs iterative RANSAC-based global registration to find the best rough alignment.
    Matches the logic of testing multiple thresholds to find the highest fitness and lowest RMSE.
    """
    max_attempts = 10
    best_fitness = -0.1
    best_inlier_rmse = 100.0
    best_result = None
    best_threshold = None 

    # Thresholds to test, from coarse (10.0) to fine (3.0)
    thresholds = [10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]
    # thresholds = [7]

    for attempt in range(max_attempts):
        for thr in thresholds:            
            result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
                source_down, target_down, source_fpfh, target_fpfh, 
                mutual_filter=True, # Improved matching
                max_correspondence_distance=thr,
                estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
                ransac_n=3, 
                checkers=[
                    o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.85),
                    o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(thr)
                ], 
                criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(10000, 0.99)
            )

            # Update best result if fitness is good and RMSE is lower
            if result.fitness > 0.85 and result.inlier_rmse < best_inlier_rmse:
                best_fitness = result.fitness
                best_inlier_rmse = result.inlier_rmse
                best_result = result
                best_threshold = thr
            
            # Early exit if we find a very high-quality match
            if best_fitness > 0.95 and best_inlier_rmse < 2.35: 
                print(f"Excellent Global Fit Found at Threshold {best_threshold}")
                return best_result, best_threshold
            
    if best_result is None:
        print("Warning: RANSAC could not find a fit above 0.85 fitness.")
        # Fallback to the last result generated if nothing met the 0.85 criteria
        return result, thr

    print(f"RANSAC Finished. Best Threshold: {best_threshold} | Fitness: {best_fitness:.4f}")
    return best_result, best_threshold


# One time local refinement using ICP
def run_local_refinement(source, target, initial_transformation, voxel_size):
    """
    Performs ICP registration to refine the alignment found by RANSAC.
    """
    # We use a smaller threshold for ICP to ensure high precision
    distance_threshold = voxel_size * 0.4
    
    result = o3d.pipelines.registration.registration_icp(
        source, target, distance_threshold, initial_transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
    )
    return result


# Progressive Local Refinement using ICP
def run_local_refinement_adaptive(source, target, initial_trans, best_ransac_thr):
    """
    Refines alignment using an iterative ICP loop. 
    Starts with a coarse threshold and progressively tightens for precision.
    """
    # Define a range of multipliers to tighten the search radius
    # multipliers = [1.0, 0.8, 0.5, 0.25]
    multipliers = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]
    # multipliers = [0.6]
    thresholds = [best_ransac_thr * m for m in multipliers]
    
    best_result = None
    best_inlier_rmse = float('inf') # Start with the highest possible error
    
    print(f"{'Threshold':<12} | {'Fitness':<12} | {'RMSE':<12}")
    print("-" * 45)

    for thr in thresholds:
        # Execute Point-to-Point ICP
        reg_icp = o3d.pipelines.registration.registration_icp(
            source, target, thr, initial_trans,
            o3d.pipelines.registration.TransformationEstimationPointToPoint(),
            o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=1000)
        )

        # Logging the current iteration results
        print(f"{thr:<12.2f} | {reg_icp.fitness:<12.4f} | {reg_icp.inlier_rmse:<12.4f}")

        # Selection Logic: Prioritize lowest RMSE (highest precision) 
        # as long as the fitness is acceptable (e.g., > 85%)
        if reg_icp.fitness > 0.85 and reg_icp.inlier_rmse < best_inlier_rmse:
            best_inlier_rmse = reg_icp.inlier_rmse
            best_result = reg_icp

    # Fallback: if no run met the 85% fitness, return the last result
    return best_result if best_result is not None else reg_icp

In [102]:
# testing purpose
test_pcd = o3d.io.read_point_cloud("pcd_data/view02.pcd")
o3d.visualization.draw_geometries([test_pcd], window_name="Downsampled Clouds with Normals")

In [103]:
# Main Processing Pipeline

# --- 1. Configuration ---
DATA_DIR = "pcd_data"
SOURCE_DIR = "viewpoints_candidate"
SOURCE_PATH = "workpiece/first_object_10000.pcd" #  CAD model

tf_obj = np.load(r'pcd_data/initial_obj_pose.npy')
YOLO_POS = tf_obj[:3]                                   # Initial guess from YOLO
YOLO_ANGLE = tf_obj[4]                                  # Initial angle from YOLO
CROP_BOX = [95, 75, 60]                               # Region of interest size


# --- 2. Data Preparation ---
print("Step 1: Loading all point clouds...")
target_cloud, source_cloud = load_viewpoint_data(2, DATA_DIR, SOURCE_DIR)
target_cloud = process_point_cloud(target_cloud, YOLO_POS, YOLO_ANGLE, CROP_BOX)

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0, origin=[0, 0, 0])
# o3d.visualization.draw_geometries([target_cloud, source_cloud, world_frame], window_name="Processed Target Cloud")

o3d.visualization.draw_geometries([target_cloud], window_name="Processed Target Cloud")

# # 3. Preprocess Normal
source_down, source_fpfh = preprocess_normal(source_cloud)
target_down, target_fpfh = preprocess_normal(target_cloud, invert_normals=True)
# o3d.visualization.draw_geometries([target_down, source_down, world_frame], window_name="Processed Target Cloud")


Step 1: Loading all point clouds...


In [67]:
# --- 3. Global Alignment (RANSAC) ---
print("Step 2: Running RANSAC Global Registration...")
ransac_res, best_thr = run_global_registration_adaptive(source_down, target_down, source_fpfh, target_fpfh)
print(ransac_res)

# Visualize RANSAC result
source_temp = copy.deepcopy(source_cloud)
source_temp.transform(ransac_res.transformation)
source_temp.paint_uniform_color([1, 0, 0])
target_down.paint_uniform_color([0, 0.651, 0.929])
o3d.visualization.draw_geometries([source_temp, target_down], window_name="RANSAC Result")

Step 2: Running RANSAC Global Registration...
[Open3D WARNING] Too few correspondences (86) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (86) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (86) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (86) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (86) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (86) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (86) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (86) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (86) after mutual filter, fall back to original correspondences.


In [63]:
# --- 4. Local Alignment (ICP) ---
print("Step 3: Running ICP Local Refinement...")
icp_res = run_local_refinement_adaptive(source_cloud, target_down, ransac_res.transformation, best_thr)
print(icp_res)

# --- 5. Extract Final Results ---
final_matrix = icp_res.transformation
initial_POS = final_matrix[:3, 3] 
initial_ORI = final_matrix[:3, :3] 
print("="*30)
print(f"Final Position: {initial_POS}")
print(f"Final Orientation Matrix:\n{initial_ORI}")
print(f"Fitness: {icp_res.fitness:.4f}")
print(f"RMSE: {icp_res.inlier_rmse:.4f}")


# --- 6. Final Visualization ---
source_final = copy.deepcopy(source_cloud).transform(final_matrix)
source_final.paint_uniform_color([1, 0, 0])         # Red: CAD Model
target_cloud.paint_uniform_color([0, 0.65, 0.93])   # Blue: Scanned Data
o3d.visualization.draw_geometries([source_final, target_cloud])

Step 3: Running ICP Local Refinement...
Threshold    | Fitness      | RMSE        
---------------------------------------------
3.00         | 0.1554       | 1.7643      
2.70         | 0.1385       | 1.5828      
2.40         | 0.1257       | 1.4536      
2.10         | 0.1080       | 1.2846      
1.80         | 0.0911       | 1.1129      
1.50         | 0.0775       | 0.9847      
1.20         | 0.0572       | 0.8384      
0.90         | 0.0294       | 0.6412      
0.60         | 0.0139       | 0.4230      
0.30         | 0.0026       | 0.1657      
RegistrationResult with fitness=2.634550e-03, inlier_rmse=1.656539e-01, and correspondence_set size of 7
Access transformation to get result.
Final Position: [556.67910768 -50.30841964   4.1563991 ]
Final Orientation Matrix:
[[ 0.75439671  0.60906352 -0.24480039]
 [ 0.62731979 -0.77874973 -0.00433033]
 [-0.19327568 -0.15030134 -0.96956383]]
Fitness: 0.0026
RMSE: 0.1657
